# TideTrace training

**SIH26143** &middot; NTRO oil spill attribution &middot; model stage

Three class segmentation of Sentinel-1 SAR: sea, look-alike, mineral oil.

**Settings:** Accelerator *GPU P100* (or T4 x2), Internet *On*.
**Secrets:** `HF_TOKEN`.

## The point of the checkpointing

Kaggle kills a session at its time limit and takes the disk with it. So this
pushes the best checkpoint to the Hub every time validation IoU improves, and
stops cleanly a little before the limit. Re-run with `RESUME = True` and it
picks up from the Hub. A twelve hour budget becomes as many sessions as you
need.

Kaggle is the factory. The laptop is the product. Never serve the judged demo
from here.


## 1. Environment


In [ ]:
!pip -q install segmentation-models-pytorch albumentations huggingface_hub 2>&1 | tail -2
import os, sys, subprocess, json
from pathlib import Path

def sh(*args):
    print('$', ' '.join(args), flush=True)
    subprocess.run(list(args), check=True)

# No credential is required to run this notebook. The TideTrace repos on the
# Hub are public, so code downloads anonymously, and results leave through
# Kaggle's own kernel output. A token is used only if you have chosen to add
# one as a Secret, in which case results are ALSO mirrored to the Hub.
HAVE_HF_TOKEN = False
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    HAVE_HF_TOKEN = True
    print('HF token found: results will also be mirrored to the Hub')
except Exception:
    print('No HF_TOKEN secret. Downloads are anonymous and results leave via')
    print('kernel output. This is the normal path and nothing is missing.')

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print('%s, %.0f GB' % (p.name, p.total_memory / 1e9))
    # Fail here, not eleven minutes later inside the first forward
    # pass. Kaggle ships torch built for sm_70 and up; the P100 is
    # Pascal, sm_60, and every conv on it raises 'no kernel image is
    # available for execution on the device' -- after the archive has
    # been downloaded and the tiles have been staged.
    cap = p.major * 10 + p.minor
    builds = torch.cuda.get_arch_list()
    print('capability sm_%d | torch builds %s' % (cap, ' '.join(builds)))
    if ('sm_%d' % cap) not in builds:
        raise SystemExit(
            'This GPU is sm_%d and this torch has no kernels for it. Use '
            'T4 (sm_75): kaggle_push.py train --accelerator NvidiaTeslaT4, '
            'or change the accelerator in the Kaggle editor.' % cap)


## 2. Pull the source


In [ ]:
from huggingface_hub import snapshot_download

CODE_DIR = Path('/kaggle/working/tidetrace')
snapshot_download(repo_id='N-1ACE/tidetrace-oil-unet', repo_type='model',
                  local_dir=str(CODE_DIR), allow_patterns=['code/**'])
sys.path.insert(0, str(CODE_DIR / 'code'))

# Keep every path Kaggle-local. The package writes under data/ by default.
os.environ['TIDETRACE_DATA'] = '/kaggle/working/data'
os.environ['TIDETRACE_MODELS'] = '/kaggle/working/models'

from app import config, hub
print('TideTrace', config.VERSION, '| tile', config.TILE, '| overlap', config.TILE_OVERLAP)
print('dataset repo:', hub.DATASET_REPO)
print('model repo  :', hub.MODEL_REPO)


## 3. Find the tiles

First choice is the chained kernel output: this notebook declares
`tidetrace-prepare-data` as a source, so its tiles appear under
`/kaggle/input` with no download and no credential. The Hub is the fallback
for training from a previously published tile set.


In [ ]:
SANITY = False   # True uses a small subset for a five minute smoke run

roots = sorted(Path('/kaggle/input').glob('*/tiles'))
roots += sorted(Path('/kaggle/input').glob('*/kaggle/working/tiles'))
roots += [p for p in sorted(Path('/kaggle/input').glob('*'))
          if (p / 'index.json').exists()]
TILES = roots[0] if roots else None

if TILES is not None:
    print('using chained kernel output:', TILES)
else:
    print('no chained input found; falling back to the Hub')
    TILES = Path(hub.pull_tiles(Path('/kaggle/temp/tiles'),
                               max_files=200 if SANITY else None))

# Stop here, loudly, rather than three cells later inside the class
# balance check. This notebook cannot invent training data.
_n = len(sorted(Path(TILES).glob('*.npz'))) if TILES else 0
if _n == 0:
    raise SystemExit(
        'No tiles are available. This notebook trains on the output of '
        'tidetrace-prepare-data, which must finish successfully first. '
        'Check its status before running this one.')
print('%d tiles available' % _n)
files = sorted(TILES.glob('*.npz'))
print('%d tiles, %.2f GB' % (len(files), sum(f.stat().st_size for f in files) / 1e9))

idx_path = TILES / 'index.json'
if idx_path.exists():
    print(json.dumps(json.loads(idx_path.read_text()), indent=2)[:800])
else:
    print('WARNING: no index.json, so default dB normalisation will be used.')


## 4. Class balance

Check this before spending GPU hours. If the oil class is a rounding error,
the fix is the sampler, not the learning rate.


In [ ]:
import numpy as np
px = np.zeros(3, dtype=np.int64)
for f in files[:300]:
    px += np.bincount(np.load(f)['label'].ravel(), minlength=3)
share = px / max(px.sum(), 1)
for i, name in enumerate(['sea', 'look-alike', 'oil']):
    print('%-12s %10d px  %6.3f%%' % (name, px[i], 100 * share[i]))
assert px[2] > 0, 'no oil pixels in the sample: check the data stage'


## 5. Train

Batch 8 on P100, 4 on a single T4. `TIME_BUDGET` should sit comfortably
under the session limit so the final push happens before the kill.


In [ ]:
from app.ml import train as train_mod

EPOCHS      = 40
RESUME      = False        # True on a continuation session

# Batch size follows the card rather than a guess. A T4 is 16 GB but
# slower and tighter under AMP than the bigger cards, so the
# spec's 4 applies there. Anything larger gets the headroom it has.
# The P100 branch is kept only so the number is right if Kaggle ever
# ships a torch that supports sm_60 again; today it cannot run at all.
BATCH, GPU = 4, 'cpu'
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    GPU = props.name
    gb = props.total_memory / 1e9
    if 'T4' in GPU:        BATCH = 4
    elif 'P100' in GPU:    BATCH = 8
    elif gb >= 38:         BATCH = 16
    elif gb >= 20:         BATCH = 12
    else:                  BATCH = 8

# Kaggle cuts GPU sessions at 9 hours. Stop well before that so the
# final push happens on our terms rather than mid-epoch.
TIME_BUDGET = 7.5 * 3600
print('%s  batch %d  epochs %d  budget %.1f h' % (GPU, BATCH, EPOCHS, TIME_BUDGET/3600))

report = train_mod.train(
    tile_dir=TILES,
    out_path='/kaggle/working/models/oil_unet_best.pt',
    arch='UnetPlusPlus',
    encoder='timm-efficientnet-b0',
    epochs=EPOCHS,
    batch_size=BATCH,
    lr=1e-4,
    workers=2,
    push_to_hub=HAVE_HF_TOKEN,
    resume=RESUME,
    time_budget_s=TIME_BUDGET,
)


## 6. The metrics card

The trained model next to the published dB threshold baseline, on identical
validation tiles. This is the table that shows the model is doing work the
baseline cannot, and it is the one to put on the slide.


In [ ]:
best, base = report['best'], report['baseline']
rows = [('IoU oil', 'iou_oil'), ('IoU look-alike', 'iou_lookalike'),
        ('IoU sea', 'iou_sea'), ('pixel accuracy', 'pixel_accuracy')]
print('%-16s %13s %10s %9s' % ('metric', '-22 dB base', 'U-Net', 'delta'))
print('-' * 52)
for label, key in rows:
    b, m = base[key], best[key]
    print('%-16s %13.4f %10.4f %+9.4f' % (label, b, m, m - b))
print('\nepochs completed:', report['epochs_completed'], 'of', report['epochs_requested'])
if report['stopped_early']:
    print('Stopped on the time budget. Re-run this notebook with RESUME = True.')


## 7. Learning curve


In [ ]:
import matplotlib.pyplot as plt
h = report['history']
if h:
    ep = [r['epoch'] for r in h]
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(ep, [r['train_loss'] for r in h]); ax[0].set_title('train loss')
    ax[1].plot(ep, [r['iou_oil'] for r in h], label='oil')
    ax[1].plot(ep, [r['iou_lookalike'] for r in h], label='look-alike')
    ax[1].axhline(base['iou_oil'], ls='--', c='grey', label='dB baseline, oil')
    ax[1].set_title('validation IoU'); ax[1].legend()
    for a in ax: a.set_xlabel('epoch'); a.grid(alpha=.3)
    plt.tight_layout(); plt.show()


## 8. Verify the shipped artefact

The checkpoint has to stay under 80 MB, because it is the one file the demo
laptop carries.


In [ ]:
ck = Path('/kaggle/working/models/oil_unet_best.pt')
mb = ck.stat().st_size / 1e6
print('%s  %.1f MB' % (ck.name, mb))
assert mb < 80, 'checkpoint exceeds the 80 MB budget'
if HAVE_HF_TOKEN:
    print('On the Hub:', 'https://huggingface.co/' + hub.MODEL_REPO)
print('\nOn the laptop, run:')
print('    python scripts/kaggle_push.py output train --dir models')
print('    python scripts/hf_sync.py push-model    # archive it')
print('then restart the app. The top bar switches to U-NET LOADED.')
